# CAP Automation Example

This notebook demonstrates how to control CrysAlisPro programmatically using the `CAPInstance` class.

**Topics covered:**
- Starting and connecting to CAP
- Loading experiments
- Running data reduction workflows
- Batch command execution
- Error handling
- Graceful shutdown

**Requirements:** CrysAlisPro installed, example data extracted

**Audience:** CAP users who want to automate repetitive tasks

## 1. Setup

Extract example data and import the CAP control interface.

In [ ]:
import os
import zipfile
from pathlib import Path

# Navigate to repository root and extract data
if Path.cwd().name == 'examples':
    os.chdir('..')

from cap_auto.cap_control import CAPInstance

data_dir = Path('example_data/exp_11317')
zip_file = Path('example_data/exp_11317.zip')

if not data_dir.exists() and zip_file.exists():
    print(f"Extracting {zip_file}...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall('example_data')
    print(f"✓ Extracted")

# Path to experiment
par_file = str(data_dir.absolute() / 'exp_11317.par')
print(f"✓ Experiment: {par_file}")

## 2. Start CAP and Load Experiment

Create a CAP instance and load the example experiment.

In [ ]:
# Create CAP instance (will start CAP automatically)
cap = CAPInstance(par_file=par_file, start_now=True, raise_on_error=False, min_cap_version='44.100', max_cap_version='44.200')

print(f"✓ CAP started (version {cap.cap_version[0]}.{cap.cap_version[1]})")
print(f"✓ Status: {cap.get_status()}")

## 3. Execute Single Commands

Run basic CAP commands and inspect results.

In [ ]:
# Test with a simple command
result = cap.execute("xx sleep 2")

print(f"Command: {result.command}")
print(f"Success: {result.success}")
print(f"Execution time: {result.execution_time:.2f} s")
print(f"Warnings: {len(result.warnings)}")
print(f"Errors: {len(result.errors)}")

## 4. Data Reduction Workflow

Run the full autoprocessing pipeline.

In [ ]:
# Run data reduction commands
print("Running data reduction...")

# Profile fitting
print('Data reduction using template XML file for proffit parameters')
result_proffit = cap.execute("dc proffit xml exp_11317_proffit_pars", timeout=240)
print(f"✓ dc proffit xml exp_11317_proffit_pars: {result_proffit.execution_time:.1f} s")


In [ ]:

# Reciprocal refinement
print('Refinalizing (just to try it out)')
result_rrp = cap.execute("dc rrpfromxml exp_11317_0_9_A_finalizer.xml", timeout=120)
print(f"✓ dc rrpfromxml exp_11317_0_9_A_finalizer.xml: {result_rrp.execution_time:.1f} s")

## 6. Command History

Review all executed commands.

In [ ]:
# Display command history
print(f"Executed {len(cap.history)} commands:\n")

for i, cmd_result in enumerate(cap.history, 1):
    status = '✓' if cmd_result.success else '✗'
    print(f"{i}. {status} {cmd_result.command[:50]:<50} ({cmd_result.execution_time:.2f}s)")

## 7. Cleanup

Stop CAP gracefully.

In [ ]:
# Stop CAP instance
cap.stop()
print("✓ CAP stopped successfully")